In [29]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Does MyDrive exist?", drive_root.exists())

if drive_root.exists():
    matches = list(drive_root.rglob("financial_rag_rebuild.zip"))

    if matches:
        print("\nZIP file found:")
        for path in matches:
            print(path)
    else:
        print("\nZIP not found. Here are the files in MyDrive:")
        for item in list(drive_root.iterdir())[:40]:
            print(item.name)

Does MyDrive exist? True

ZIP file found:
/content/drive/MyDrive/financial_rag_rebuild.zip


In [5]:
from pathlib import Path
import zipfile
import shutil

zip_path = Path("/content/drive/MyDrive/financial_rag_rebuild.zip")
project = Path("/content/financial_rag_project")

# Start with a clean extraction directory
if project.exists():
    shutil.rmtree(project)

project.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(project)

print("Extraction completed successfully.")

Extraction completed successfully.


In [6]:
from pathlib import Path

project = Path("/content/financial_rag_project")

print("Project files:")
for item in sorted(project.iterdir()):
    print(repr(item.name))

Project files:
'build_index.py'
'raw\\alphabet(google)\\Alphabet_2021_Annual_report.pdf'
'raw\\alphabet(google)\\Alphabet_2022_Annual_Report.pdf'
'raw\\alphabet(google)\\Alphabet_2023_Annual_Report.pdf'
'raw\\alphabet(google)\\Alphabet_2024_Annual_Report.pdf'
'raw\\alphabet(google)\\Alphabet_2025_Annual_Report.pdf'
'raw\\amazon\\Amazon_2021_Annual_Report.pdf'
'raw\\amazon\\Amazon_2022_Annual_Report.pdf'
'raw\\amazon\\Amazon_2023_Annual_Report.pdf'
'raw\\amazon\\Amazon_2024_Annual_Report.pdf'
'raw\\amazon\\Amazon_2025_Annual_Report.pdf'
'raw\\apple\\Apple_2021_Annual_Report.pdf'
'raw\\apple\\Apple_2022_Annual_Report.pdf'
'raw\\apple\\Apple_2023_Annual_Report.pdf'
'raw\\apple\\Apple_2024_Annual_Report.pdf'
'raw\\apple\\Apple_2025_Annual_Report.pdf'
'raw\\microsoft\\Microsoft_2021_Annual_Report.pdf'
'raw\\microsoft\\Microsoft_2022_Annual_Report.pdf'
'raw\\microsoft\\Microsoft_2023_Annual_Report.pdf'
'raw\\microsoft\\Microsoft_2024_Annual_Report.pdf'
'raw\\microsoft\\Microsoft_2025_Annual_

In [7]:
from pathlib import Path
import shutil

project = Path("/content/financial_rag_project")
fixed_count = 0

for item in list(project.iterdir()):
    if not item.is_file() or "\\" not in item.name:
        continue

    # Convert embedded Windows backslashes into directory separators
    parts = item.name.split("\\")
    destination = project.joinpath(*parts)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if not destination.exists():
        shutil.move(str(item), str(destination))
        fixed_count += 1
    else:
        print("Skipped existing file:", destination)

print("Files moved into proper directories:", fixed_count)
print("build_index.py exists:", (project / "build_index.py").exists())
print("chunking.py exists:", (project / "src" / "chunking.py").exists())

Files moved into proper directories: 45
build_index.py exists: True
chunking.py exists: True


In [9]:
from pathlib import Path

project = Path("/content/financial_rag_project")

print("Project files:")
for name in ["build_index.py", "requirements.txt", "src/chunking.py",
             "src/embeddings.py", "src/vector_store.py"]:
    path = project / name
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

pdfs = list((project / "raw").rglob("*.pdf"))
print("\nPDF count in data/raw:", len(pdfs))

Project files:
build_index.py: OK
requirements.txt: OK
src/chunking.py: OK
src/embeddings.py: OK
src/vector_store.py: OK

PDF count in data/raw: 25


In [10]:
from pathlib import Path

project = Path("/content/financial_rag_project")

files_to_check = [
    "build_index.py",
    "src/document_loader.py",
    "src/chunking.py",
    "src/vector_store.py"
]

for filename in files_to_check:
    path = project / filename
    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if path.exists():
        print(path.read_text())
    else:
        print("FILE NOT FOUND")


==================== build_index.py ====================
from src.document_loader import DocumentLoader
from src.chunking import ChunkCreator
from src.embeddings import EmbeddingModel
from src.vector_store import VectorStore


def main():

    print("=" * 60)
    print("Building Financial RAG Knowledge Base")
    print("=" * 60)

    # ----------------------------
    # Load Documents
    # ----------------------------

    print("\nLoading Documents...")

    loader = DocumentLoader()

    documents = loader.load_documents()

    print(f"Loaded {len(documents)} documents.")

    # ----------------------------
    # Chunking
    # ----------------------------

    print("\nCreating Chunks...")

    chunker = ChunkCreator()

    chunks = chunker.create_chunks(documents)

    print(f"Created {len(chunks)} chunks.")

    # ----------------------------
    # Embedding Model
    # ----------------------------

    print("\nLoading Embedding Model...")

    embedding_loader = EmbeddingModel

In [12]:
from pathlib import Path

project = Path("/content/financial_rag_project")
loader_file = project / "src" / "document_loader.py"

code = loader_file.read_text()

old_path = 'self.data_dir = Path("data/raw")'
new_path = 'self.data_dir = Path("raw")'

if old_path in code:
    code = code.replace(old_path, new_path)
    loader_file.write_text(code)
    print("Document loader path updated to raw/")
elif new_path in code:
    print("Document loader already uses raw/")
else:
    print("Expected path not found. No changes made.")

Document loader path updated to raw/


In [13]:
from pathlib import Path

project = Path("/content/financial_rag_project")
vector_file = project / "src" / "vector_store.py"

code = vector_file.read_text()

old_path = '"vector_db"'
new_path = '"vector_db_rebuilt"'

if old_path in code:
    code = code.replace(old_path, new_path, 1)
    vector_file.write_text(code)
    print("New index path configured: data/vector_db_rebuilt/")
else:
    print("Check vector_store.py manually; no change made.")

New index path configured: data/vector_db_rebuilt/


In [14]:
from pathlib import Path

project = Path("/content/financial_rag_project")

print("Document loader:")
print((project / "src" / "document_loader.py").read_text())

print("\nVector store:")
print((project / "src" / "vector_store.py").read_text())

print("\nPDF count:", len(list((project / "raw").rglob("*.pdf"))))

Document loader:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
class DocumentLoader:
    def __init__(self):
        self.data_dir = Path("raw")
    def load_documents(self):
        documents = []
        pdf_files = list(self.data_dir.rglob("*.pdf"))
        if not pdf_files:
            raise FileNotFoundError(
                f"No PDF files found in {self.data_dir}"
            )
        for pdf_file in pdf_files:
            loader = PyPDFLoader(str(pdf_file))
            docs = loader.load()

           # Extract Company & Year
            
            filename = pdf_file.stem
            parts = filename.split("_")
            company = parts[0]
            year = int(parts[1])
            
            # Add Metadata

            for doc in docs:
                page = doc.metadata.get("page", 0)
                doc.metadata = {
                        "company": company,
                        "year": year,
                        "sourc

In [15]:
from pathlib import Path

project = Path("/content/financial_rag_project")

loader_file = project / "src" / "document_loader.py"
vector_file = project / "src" / "vector_store.py"

print("Document loader uses raw/:", 'Path("raw")' in loader_file.read_text())
print("New FAISS directory configured:", '"vector_db_rebuilt"' in vector_file.read_text())
print("PDF count:", len(list((project / "raw").rglob("*.pdf"))))

Document loader uses raw/: True
New FAISS directory configured: True
PDF count: 25


In [16]:
from pathlib import Path

project = Path("/content/financial_rag_project")
print((project / "requirements.txt").read_text())

langchain
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-google-genai
sentence-transformers
faiss-cpu
pypdf
python-dotenv
torch
transformers
streamlit


In [17]:
%cd /content/financial_rag_project
%pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-google-genai sentence-transformers faiss-cpu pypdf python-dotenv transformers streamlit pymupdf

/content/financial_rag_project
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [18]:
import torch
import langchain
import transformers
import sentence_transformers
import faiss
import fitz

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("LangChain:", langchain.__version__)
print("Transformers:", transformers.__version__)
print("All core imports successful!")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
LangChain: 1.4.7
Transformers: 5.20.2
All core imports successful!


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [19]:
%cd /content/financial_rag_project

from src.document_loader import DocumentLoader

loader = DocumentLoader()
documents = loader.load_documents()

print("Documents loaded successfully:", len(documents))
print("Sample metadata:", documents[0].metadata)
print("Sample text:", documents[0].page_content[:300])

/content/financial_rag_project
Documents loaded successfully: 2840
Sample metadata: {'company': 'Microsoft', 'year': 2024, 'source': 'Microsoft_2024_Annual_Report.pdf', 'page': 0}
Sample text: 


In [20]:
empty_pages = sum(
    1 for doc in documents
    if not doc.page_content.strip()
)

print("Total pages:", len(documents))
print("Pages with text:", len(documents) - empty_pages)
print("Empty pages:", empty_pages)
print("\nSample non-empty text:")

for doc in documents:
    if doc.page_content.strip():
        print(doc.page_content[:500])
        break

Total pages: 2840
Pages with text: 2803
Empty pages: 37

Sample non-empty text:
1 
Dear shareholders, colleagues, customers, and partners: 
Fiscal year 2024 was a pivotal year for Microsoft. We entered our 50th year as a company and the second year of the AI 
platform shift. With these milestones, I’ve found myself reflecting on how Microsoft has remained a consequential company 
decade after decade in an industry with no franchise value. And I realize that it’s because—time and time again, when tech 
paradigms have shifted —we have seized the opportunity to reinvent oursel


In [21]:
from src.chunking import ChunkCreator

chunker = ChunkCreator(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    chunk_size=200,
    chunk_overlap=30
)

chunks = chunker.create_chunks(documents)

print("Total chunks created:", len(chunks))
print("\nSample chunk:")
print(chunks[0].page_content[:500])
print("\nSample metadata:")
print(chunks[0].metadata)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (561 > 512). Running this sequence through the model will result in indexing errors


Total chunks created: 12686

Sample chunk:
1 
Dear shareholders, colleagues, customers, and partners: 
Fiscal year 2024 was a pivotal year for Microsoft. We entered our 50th year as a company and the second year of the AI 
platform shift. With these milestones, I’ve found myself reflecting on how Microsoft has remained a consequential company 
decade after decade in an industry with no franchise value. And I realize that it’s because—time and time again, when tech 
paradigms have shifted —we have seized the opportunity to reinvent oursel

Sample metadata:
{'company': 'Microsoft', 'year': 2024, 'source': 'Microsoft_2024_Annual_Report.pdf', 'page': 1}


In [22]:
import pickle
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

save_dir = Path("/content/drive/MyDrive/financial_rag_checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)

chunks_file = save_dir / "financial_chunks.pkl"

with open(chunks_file, "wb") as f:
    pickle.dump(chunks, f)

print("Chunks saved to:", chunks_file)
print("Total chunks saved:", len(chunks))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chunks saved to: /content/drive/MyDrive/financial_rag_checkpoints/financial_chunks.pkl
Total chunks saved: 12686


In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

lengths = [
    len(tokenizer.encode(
        chunk.page_content,
        add_special_tokens=True,
        truncation=False
    ))
    for chunk in chunks
]

print("Total chunks:", len(chunks))
print("Chunks within 256 tokens:", sum(n <= 256 for n in lengths))
print("Chunks exceeding 256 tokens:", sum(n > 256 for n in lengths))
print("Longest chunk:", max(lengths))

Total chunks: 12686
Chunks within 256 tokens: 12686
Chunks exceeding 256 tokens: 0
Longest chunk: 202


In [24]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

checkpoint = Path(
    "/content/drive/MyDrive/financial_rag_checkpoints/financial_chunks.pkl"
)

print("Chunk checkpoint exists:", checkpoint.exists())

if checkpoint.exists():
    print("Checkpoint size (MB):", round(checkpoint.stat().st_size / (1024 * 1024), 2))

print("\nYour notebook should also be saved in Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chunk checkpoint exists: True
Checkpoint size (MB): 11.01

Your notebook should also be saved in Google Drive.


In [25]:
from src.embeddings import EmbeddingModel

embedding_loader = EmbeddingModel()
embedding_model = embedding_loader.load_model()

print("Embedding model loaded successfully!")

Using embedding device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [26]:
import pickle
from pathlib import Path
from google.colab import drive
from src.vector_store import VectorStore

drive.mount("/content/drive")

# Restore chunks from the saved checkpoint
checkpoint = Path(
    "/content/drive/MyDrive/financial_rag_checkpoints/financial_chunks.pkl"
)

with open(checkpoint, "rb") as f:
    chunks = pickle.load(f)

print("Chunks restored:", len(chunks))

# Create the FAISS index
vector_store = VectorStore()
vector_db = vector_store.create_vector_db(chunks, embedding_model)

print("FAISS index created:", vector_db.index.ntotal)

# Save to the separate rebuilt index directory
vector_store.save_vector_db(vector_db)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chunks restored: 12686
FAISS index created: 12686
Vector Database Saved Successfully!


In [27]:
from pathlib import Path

project = Path("/content/financial_rag_project")
index_dir = project / "data" / "vector_db_rebuilt"

print("Index directory exists:", index_dir.exists())

for filename in ["index.faiss", "index.pkl"]:
    file_path = index_dir / filename
    print(
        f"{filename}:",
        "Saved" if file_path.exists() else "Missing"
    )
    if file_path.exists():
        print(f"  Size: {file_path.stat().st_size / (1024 * 1024):.2f} MB")

Index directory exists: True
index.faiss: Saved
  Size: 18.58 MB
index.pkl: Saved
  Size: 11.65 MB


In [28]:
from pathlib import Path
import shutil

source = Path(
    "/content/financial_rag_project/data/vector_db_rebuilt"
)

destination = Path(
    "/content/drive/MyDrive/financial_rag_vector_db_rebuilt"
)

destination.mkdir(parents=True, exist_ok=True)

for filename in ["index.faiss", "index.pkl"]:
    shutil.copy2(source / filename, destination / filename)

print("Index backup completed!")

for filename in ["index.faiss", "index.pkl"]:
    file_path = destination / filename
    print(f"{filename}: {file_path.stat().st_size / (1024 * 1024):.2f} MB")

print("Drive location:", destination)

Index backup completed!
index.faiss: 18.58 MB
index.pkl: 11.65 MB
Drive location: /content/drive/MyDrive/financial_rag_vector_db_rebuilt


In [30]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

index_dir = Path(
    "/content/drive/MyDrive/financial_rag_vector_db_rebuilt"
)

print("Index directory exists:", index_dir.exists())

for filename in ["index.faiss", "index.pkl"]:
    file_path = index_dir / filename
    print(f"{filename} exists:", file_path.exists())

project_dir = Path("/content/financial_rag_project")
print("Project files exist:", project_dir.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index directory exists: True
index.faiss exists: True
index.pkl exists: True
Project files exist: True


In [31]:
from pathlib import Path
import torch

project = Path("/content/financial_rag_project")

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nRequired project files:")

for filename in [
    "src/embeddings.py",
    "src/vector_store.py",
    "src/retriever.py",
    "src/rag_chain.py",
    "src/llm.py",
]:
    path = project / filename
    print(f"{filename}: {'Found' if path.exists() else 'Missing'}")

print("\nSaved FAISS index:")
index_dir = Path(
    "/content/drive/MyDrive/financial_rag_vector_db_rebuilt"
)

for filename in ["index.faiss", "index.pkl"]:
    print(f"{filename}: {(index_dir / filename).exists()}")

GPU available: True
GPU: Tesla T4

Required project files:
src/embeddings.py: Found
src/vector_store.py: Found
src/retriever.py: Found
src/rag_chain.py: Found
src/llm.py: Found

Saved FAISS index:
index.faiss: True
index.pkl: True


In [32]:
%cd /content/financial_rag_project

from src.embeddings import EmbeddingModel
from src.vector_store import VectorStore

# Load the embedding model
embedding_loader = EmbeddingModel()
embedding_model = embedding_loader.load_model()

print("Embedding model loaded.")

# Point to the saved index in Google Drive
vector_store = VectorStore()
vector_store.vector_db_path = (
    __import__("pathlib").Path(
        "/content/drive/MyDrive/financial_rag_vector_db_rebuilt"
    )
)

# Load the existing FAISS index
vector_db = vector_store.load_vector_db(embedding_model)

print("FAISS index loaded successfully.")
print("Total vectors:", vector_db.index.ntotal)

/content/financial_rag_project
Using embedding device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
FAISS index loaded successfully.
Total vectors: 12686


In [33]:
query = "What were Microsoft's major business developments in fiscal year 2024?"

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

results = retriever.invoke(query)

print("Query:", query)
print("\nRetrieved documents:", len(results))

for i, doc in enumerate(results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Result {i}")
    print("Company:", doc.metadata.get("company"))
    print("Year:", doc.metadata.get("year"))
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("\nContent:")
    print(doc.page_content[:800])

Query: What were Microsoft's major business developments in fiscal year 2024?

Retrieved documents: 4

Result 1
Company: Microsoft
Year: 2025
Source: Microsoft_2025_Annual_Report.pdf
Page: 20

Content:
December 3, 2024.  
OVERVIEW  
Microsoft is a technology company committed to making digital technology and artificial intelligence (“AI”) available broadly 
and doing so responsibly, with a mission to empower every person and every organization on the planet to achieve more. 
We create p latforms and tools, powered by AI, that deliver innovative solutions that meet the evolving needs of our 
customers.  
We generate revenue by offering a wide range of cloud -based solutions, content, and other services to people and 
businesses; licensing and supporting an array of software products; delivering relevant online advertising to a global 
audience; and designing and selling devices. Our most significant expenses are related to compensating employees; 
supporting and investing in our cloud-b

In [34]:
from langchain_core.documents import Document

all_docs = vector_db.docstore._dict.values()

microsoft_2024_docs = [
    doc for doc in all_docs
    if doc.metadata.get("company") == "Microsoft"
    and str(doc.metadata.get("year")) == "2024"
]

print("Microsoft FY2024 chunks:", len(microsoft_2024_docs))

for doc in microsoft_2024_docs[:3]:
    print("\nSource:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:500])

Microsoft FY2024 chunks: 349

Source: Microsoft_2024_Annual_Report.pdf
Page: 1
1 
Dear shareholders, colleagues, customers, and partners: 
Fiscal year 2024 was a pivotal year for Microsoft. We entered our 50th year as a company and the second year of the AI 
platform shift. With these milestones, I’ve found myself reflecting on how Microsoft has remained a consequential company 
decade after decade in an industry with no franchise value. And I realize that it’s because—time and time again, when tech 
paradigms have shifted —we have seized the opportunity to reinvent oursel

Source: Microsoft_2024_Annual_Report.pdf
Page: 1
mission to empower every person and every organization on the planet to achieve more.  
This year, we moved from talking about AI to helping our customers translate it into real outcomes —one person, one 
organization, one institution, and one country at a time. We have made remarkable progress on this front across every 
industry. For example:  
• Coles is generating

In [35]:
from langchain_community.vectorstores import FAISS

# Collect Microsoft FY2024 chunks
all_docs = list(vector_db.docstore._dict.values())

microsoft_2024_docs = [
    doc for doc in all_docs
    if doc.metadata.get("company") == "Microsoft"
    and str(doc.metadata.get("year")) == "2024"
]

# Create a temporary FAISS index using only these chunks
filtered_vector_db = FAISS.from_documents(
    documents=microsoft_2024_docs,
    embedding=embedding_model
)

print("Filtered index created.")
print("Total chunks:", filtered_vector_db.index.ntotal)

Filtered index created.
Total chunks: 349


In [36]:
query = "What were Microsoft's major business developments in fiscal year 2024?"

retriever = filtered_vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Result {i}")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("\nContent:")
    print(doc.page_content[:1000])


Result 1
Source: Microsoft_2024_Annual_Report.pdf
Page: 1

Content:
1 
Dear shareholders, colleagues, customers, and partners: 
Fiscal year 2024 was a pivotal year for Microsoft. We entered our 50th year as a company and the second year of the AI 
platform shift. With these milestones, I’ve found myself reflecting on how Microsoft has remained a consequential company 
decade after decade in an industry with no franchise value. And I realize that it’s because—time and time again, when tech 
paradigms have shifted —we have seized the opportunity to reinvent ourselves to stay relevant to our customers, our 
partners, and our employees. And that’s what we are doing again today.  
Microsoft has been a platform and tools company from the start. We were founded in 1975 with a belief in creating 
technology that would enable others to create their own. And, nearly 50 years later, this belief remains at the heart of our 
mission to empower every person and every organization on the planet to a

In [37]:
query = """
Microsoft fiscal year 2024 business highlights:
revenue growth, Azure and cloud services, AI products,
Microsoft Copilot, operating income, and business segments.
"""

retriever = filtered_vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Result {i}")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1200])


Result 1
Page: 26
OVERVIEW  
Microsoft is a technology company committed to making digital technology and artificial intelligence (“AI”) available broadly 
and doing so responsibly, with a mission to empower every person and every organization on the planet to achieve more. 
We create p latforms and tools, powered by AI, that deliver innovative solutions that meet the evolving needs of our 
customers.  
We generate revenue by offering a wide range of cloud -based solutions, content, and other services to people and 
businesses; licensing and supporting an array of software products; delivering relevant online advertising to a global 
audience; and designing and selling devices. Our most significant expenses are related to compensating employees; 
supporting and investing in our cloud-based services, including datacenter operations; designing, manufacturing, marketing, 
and selling our other products and services; and income taxes.  
Highlights from fiscal year 2024 compared with fisca